In [1]:
import pandas as pd
import numpy as np

In [15]:
data = pd.read_csv('../临时文件/最近一次抽卡记录.csv',index_col=False)

In [16]:
data.columns  = ['cdid','sts','date_utc0','cost_item','item_list','other']
# 筛选时间大于2025-02-19 10:40 的数据
# data = data[data['date_utc0']>='2025-02-19 10:40:00']

In [17]:
# 拆分行，将多抽获取道具拆分为单抽
res = data.drop('item_list', axis=1).join(data['item_list'].str.split(', ', expand=True).stack().reset_index(level=1, drop=True).rename('item_info'))

In [18]:
# 修改cost_num 为1
res['cost_item']  = 1
# 通过cdid 、sts进行排序
res = res.sort_values(by=['cdid','sts'],ascending=[True,True])
# 统计单用户抽卡累计次数
res['gacha_num'] = res.groupby(by='cdid')['cost_item'].transform('cumsum')
# 统计用户总计抽卡次数
res['total_gacha'] = res.groupby(by='cdid')['cost_item'].transform('sum')

In [19]:
# 判断是否抽中特定item
res['get_ssr'] = res.apply(lambda x:1 if ('504002' in x['item_info'] or '504004' in x['item_info']) else 0 ,axis=1)

In [20]:
res.query('get_ssr==1&gacha_num<=70').to_excel('../临时文件/中途出货用户.xlsx')